# 14_03_OpenTelemetry_で_RAG＋エージェ

コード：`ch14/blocks/14_03_OpenTelemetry_で_RAG＋エージェ.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


## 本文の掲載コード（参照）


```python
import os, hashlib
CAPTURE_CONTENT = os.getenv("GENAI_CAPTURE_CONTENT", "false") == "true"

def digest(s: str) -> str:
    """照合用の指紋。候補の推測・再計算は可能なので機密保護の代用にはしない"""
    return hashlib.sha256(s.encode()).hexdigest()[:16]

def call_llm(messages, model="claude-sonnet-4-5", temperature=0.2):
    with tracer.start_as_current_span("chat") as sp:
        sp.set_attribute("gen_ai.operation.name", "chat")
        sp.set_attribute("gen_ai.provider.name", "anthropic")    # Bedrock 経由なら "aws.bedrock"
        sp.set_attribute("gen_ai.request.model", model)
        sp.set_attribute("gen_ai.request.temperature", temperature)
        sp.set_attribute("prompt.sha256", digest(str(messages)))  # ★ 本文でなく指紋
        if CAPTURE_CONTENT:
            sp.set_attribute("gen_ai.input.messages", str(messages)[:8000])
        try:
            r = client.messages.create(model=model, messages=messages,
                                       temperature=temperature, max_tokens=1024)
        except Exception as e:
            sp.set_attribute("error.type", type(e).__name__)
            raise      # 例外は with を抜けるときに自動で記録され、状態も ERROR になる
        sp.set_attribute("gen_ai.usage.input_tokens", r.usage.input_tokens)
        sp.set_attribute("gen_ai.usage.output_tokens", r.usage.output_tokens)
        sp.set_attribute("gen_ai.response.finish_reasons", [r.stop_reason])
        return r
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "14", "--name=otel", "--skip-docker"], cwd=CODE, check=True)
